<div style="text-align: center; padding: 30px 0;">
  <h1 style="font-size: 36px; color: #2E86AB;">🌿 Leaf Disease Detection System</h1>
  <h2 style="color: #666;">Using Deep Learning & Computer Vision</h2>
  <p style="font-size: 18px; color: #888;"><strong>CodeAlpha Data Science Internship Project</strong></p>
  <hr style="width: 50%; border: 2px solid #2E86AB;">
</div>

---

## 📋 Complete Pipeline Overview

| Phase | Description |
|-------|-------------|
| **Phase 1** | Dataset Investigation - Auto-discover classes, check integrity |
| **Phase 2** | Exploratory Data Analysis - Visualizations & insights |
| **Phase 3** | Data Preprocessing - Cleaning, resizing, normalization |
| **Phase 4** | Train/Val/Test Split - Stratified 70/15/15 |
| **Phase 5** | Custom CNN Architecture - From-scratch design |
| **Phase 6** | Transfer Learning Models - MobileNetV2, EfficientNet, ResNet50 |
| **Phase 7** | Model Training - Early stopping, LR scheduling, checkpointing |
| **Phase 8** | Model Evaluation - Metrics, confusion matrices, ROC curves |
| **Phase 9** | Error Analysis - Misclassification investigation |
| **Phase 10** | Inference System - Predict disease from new images |

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📦 IMPORTS & SETUP
# ═══════════════════════════════════════════════════════════════════════
import os, sys, json, time, hashlib, warnings, itertools
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_curve, auc, roc_auc_score)

# Try to set plot style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')

print('✅ All imports loaded successfully')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU available: {len(gpus)} GPU(s)')
else:
    print('ℹ️  No GPU found, using CPU')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🗂️ CONFIGURATION & PATHS
# ═══════════════════════════════════════════════════════════════════════

PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

# ─── Dataset Paths ───────────────────────────────────────────────────────
# Try multiple possible locations for the dataset
POSSIBLE_DATA_DIRS = [
    PROJECT_ROOT.parent / 'archive' / 'PlantVillage' / 'PlantVillage',
    PROJECT_ROOT / 'archive' / 'PlantVillage' / 'PlantVillage',
    Path('..') / 'archive' / 'PlantVillage' / 'PlantVillage',
    Path('.') / 'archive' / 'PlantVillage' / 'PlantVillage',
    Path('.') / 'PlantVillage',
]

DATA_DIR = None
for d in POSSIBLE_DATA_DIRS:
    if d.exists() and any(d.iterdir()):
        DATA_DIR = d
        break

if DATA_DIR is None:
    # Try to find it anywhere
    for p in [PROJECT_ROOT, PROJECT_ROOT.parent, Path.cwd()]:
        matches = list(p.rglob('PlantVillage'))
        for m in matches:
            if m.is_dir() and any(m.iterdir()):
                DATA_DIR = m
                break
        if DATA_DIR:
            break

if DATA_DIR is None:
    print('⚠️  Could not find PlantVillage dataset automatically.')
    print('Please set DATA_DIR manually to your dataset location.')
else:
    print(f'✅ Dataset found at: {DATA_DIR}')

# ─── Output Paths ────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
PLOTS_DIR = RESULTS_DIR / 'plots'
LOGS_DIR = RESULTS_DIR / 'logs'
for d in [RESULTS_DIR, MODELS_DIR, PLOTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Image & Training Parameters ──────────────────────────────────────
IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS = 224, 224, 3
IMG_SIZE = (IMG_WIDTH, IMG_HEIGHT)
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
BATCH_SIZE = 32
EPOCHS = 50  # Reduce for demo; increase for full training
INITIAL_LR = 1e-3
MIN_LR = 1e-6
RANDOM_SEED = 42

# Split ratios
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

# Early stopping
EARLY_STOPPING_PATIENCE = 10
REDUCE_LR_PATIENCE = 5
REDUCE_LR_FACTOR = 0.5

print(f'  Image Size: {IMG_SIZE}')
print(f'  Batch Size: {BATCH_SIZE}')
print(f'  Max Epochs: {EPOCHS}')

---
## 🔬 PHASE 1: DATASET INVESTIGATION

Auto-discovering the PlantVillage dataset structure, classes, and data quality.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 1: DATASET INVESTIGATION
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 1: DATASET INVESTIGATION')
print('█'*70)

# Discover classes automatically
class_names = sorted([d for d in os.listdir(DATA_DIR) if (DATA_DIR / d).is_dir()])
num_classes = len(class_names)
print(f'\nFound {num_classes} classes:')

# Collect class counts and file paths
class_counts = {}
file_paths = {}
for cls in class_names:
    cls_dir = DATA_DIR / cls
    files = sorted([f for f in os.listdir(cls_dir) if (cls_dir / f).is_file()])
    file_paths[cls] = [cls_dir / f for f in files]
    class_counts[cls] = len(files)
    print(f'  • {cls}: {len(files)} images')

total_images = sum(class_counts.values())
print(f'\n📊 Total images: {total_images:,}')

# Identify healthy vs diseased
healthy_classes = [c for c in class_names if 'healthy' in c.lower()]
diseased_classes = [c for c in class_names if c not in healthy_classes]
print(f'✅ Healthy classes ({len(healthy_classes)}): {healthy_classes}')
print(f'⚠️  Diseased classes ({len(diseased_classes)}): {diseased_classes}')

healthy_count = sum(class_counts[c] for c in healthy_classes)
diseased_count = sum(class_counts[c] for c in diseased_classes)
print(f'  Healthy images: {healthy_count:,} ({healthy_count/total_images*100:.1f}%)')
print(f'  Diseased images: {diseased_count:,} ({diseased_count/total_images*100:.1f}%)')

# Check image formats, dimensions, corruption, duplicates
print('\n🔍 Checking image integrity...')
formats = Counter()
dimensions = []
corrupted = []
seen_hashes = {}
duplicates = []

for cls in tqdm(class_names, desc='Scanning classes'):
    for fpath in file_paths[cls]:
        try:
            with Image.open(fpath) as img:
                img.load()
                formats[img.format or 'Unknown'] += 1
                dimensions.append(img.size)
                
                # Check duplicates via MD5 hash
                with open(fpath, 'rb') as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()
                if file_hash in seen_hashes:
                    duplicates.append((fpath.name, seen_hashes[file_hash]))
                else:
                    seen_hashes[file_hash] = fpath.name
        except Exception as e:
            corrupted.append((str(fpath), str(e)))

print(f'\n📋 Dataset Investigation Summary:')
print(f'  Image formats: {dict(formats)}')
print(f'  Corrupted images: {len(corrupted)}')
print(f'  Duplicate images: {len(duplicates)}')

if dimensions:
    widths = [d[0] for d in dimensions]
    heights = [d[1] for d in dimensions]
    print(f'  Width range: {min(widths)}–{max(widths)} px')
    print(f'  Height range: {min(heights)}–{max(heights)} px')
    print(f'  Unique dimensions: {len(set(dimensions))}')

# Dataset size
total_size_mb = sum(os.path.getsize(f) for files in file_paths.values() for f in files) / (1024*1024)
print(f'  Dataset size: {total_size_mb:.2f} MB')

# Print corrupted/duplicate details
if corrupted:
    print(f'\n⚠️  Corrupted files (first 5):')
    for f, e in corrupted[:5]:
        print(f'    - {f}: {e}')
if duplicates:
    print(f'\n⚠️  Duplicate files (first 5):')
    for d, o in duplicates[:5]:
        print(f'    - {d} duplicates {o}')
        
print('\n✅ Phase 1 complete!')

---
## 📊 PHASE 2: EXPLORATORY DATA ANALYSIS

Visualizing class distributions, sample images, dimensions, and pixel intensities.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 2: EXPLORATORY DATA ANALYSIS
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 2: EXPLORATORY DATA ANALYSIS')
print('█'*70)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 2a. Class Distribution Bar Chart
sorted_classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in sorted_classes]
colors = ['limegreen' if c in healthy_classes else 'tomato' for c in sorted_classes]

ax = axes[0]
bars = ax.barh(range(len(sorted_classes)), counts, color=colors, edgecolor='white')
ax.set_yticks(range(len(sorted_classes)))
ax.set_yticklabels([c.replace('___', '\n').replace('_', ' ') for c in sorted_classes], fontsize=8)
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
for i, (bar, count) in enumerate(zip(bars, counts)):
    ax.text(count + 15, bar.get_y() + bar.get_height()/2, f'{count:,}', va='center', fontsize=8)
legend_patches = [
    mpatches.Patch(color='limegreen', label=f'Healthy ({len(healthy_classes)})'),
    mpatches.Patch(color='tomato', label=f'Diseased ({len(diseased_classes)})'),
]
ax.legend(handles=legend_patches, loc='lower right')

# 2b. Healthy vs Diseased Pie Chart
ax = axes[1]
ax.pie([healthy_count, diseased_count],
       labels=[f'Healthy\n{healthy_count:,}', f'Diseased\n{diseased_count:,}'],
       autopct='%1.1f%%', colors=['limegreen', 'tomato'],
       startangle=90, explode=(0.05, 0.05),
       textprops={'fontsize': 12, 'fontweight': 'bold'})
ax.set_title('Healthy vs Diseased', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: class_distribution.png')

In [ ]:
# 2c. Sample Images Grid
samples_per_class = 3
n_rows = num_classes
n_cols = samples_per_class

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))

for i, cls in enumerate(class_names):
    for j in range(samples_per_class):
        ax = axes[i, j] if n_rows > 1 else axes[j]
        try:
            fpath = file_paths[cls][j % len(file_paths[cls])]
            img = Image.open(fpath)
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.axis('off')
        if j == 0:
            ax.set_title(cls.replace('___', '\n').replace('_', ' '),
                        fontsize=9, fontweight='bold', loc='left')

plt.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'sample_images_grid.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: sample_images_grid.png')

In [ ]:
# 2d. Dimension & Pixel Intensity Analysis
widths = [d[0] for d in dimensions]
heights = [d[1] for d in dimensions]
aspect_ratios = [w / h if h > 0 else 1 for w, h in dimensions]

# Sample pixel data (500 random images for speed)
all_files = [f for files in file_paths.values() for f in files]
np.random.seed(RANDOM_SEED)
sample_files = np.random.choice(all_files, min(500, len(all_files)), replace=False)

all_r, all_g, all_b = [], [], []
for fpath in tqdm(sample_files, desc='Sampling pixels'):
    try:
        img = Image.open(fpath).convert('RGB')
        arr = np.array(img, dtype=np.float32)
        all_r.extend(arr[:,:,0].flatten())
        all_g.extend(arr[:,:,1].flatten())
        all_b.extend(arr[:,:,2].flatten())
    except:
        pass

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

# Width
axes[0,0].hist(widths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].set_title(f'Width Distribution\nMean: {np.mean(widths):.0f}', fontweight='bold')
axes[0,0].set_xlabel('Width (px)')

# Height
axes[0,1].hist(heights, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0,1].set_title(f'Height Distribution\nMean: {np.mean(heights):.0f}', fontweight='bold')
axes[0,1].set_xlabel('Height (px)')

# Aspect ratio
axes[0,2].hist(aspect_ratios, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[0,2].set_title(f'Aspect Ratio\nMean: {np.mean(aspect_ratios):.2f}', fontweight='bold')
axes[0,2].set_xlabel('Aspect Ratio')

# Width vs Height
axes[0,3].scatter(widths[:1000], heights[:1000], alpha=0.3, c='purple', s=5)
axes[0,3].set_title('Width vs Height', fontweight='bold')
axes[0,3].set_xlabel('Width'); axes[0,3].set_ylabel('Height')

# Red channel
axes[1,0].hist(all_r, bins=256, color='red', alpha=0.7, range=(0,255))
axes[1,0].set_title(f'Red Channel\nMean={np.mean(all_r):.1f}', fontweight='bold')

# Green channel
axes[1,1].hist(all_g, bins=256, color='green', alpha=0.7, range=(0,255))
axes[1,1].set_title(f'Green Channel\nMean={np.mean(all_g):.1f}', fontweight='bold')

# Blue channel
axes[1,2].hist(all_b, bins=256, color='blue', alpha=0.7, range=(0,255))
axes[1,2].set_title(f'Blue Channel\nMean={np.mean(all_b):.1f}', fontweight='bold')

# Combined RGB
axes[1,3].hist(all_r, bins=256, color='red', alpha=0.4, label='R', range=(0,255))
axes[1,3].hist(all_g, bins=256, color='green', alpha=0.4, label='G', range=(0,255))
axes[1,3].hist(all_b, bins=256, color='blue', alpha=0.4, label='B', range=(0,255))
axes[1,3].set_title('Combined RGB', fontweight='bold')
axes[1,3].legend()

for ax in axes.flatten():
    ax.set_xlabel(''); ax.set_ylabel('')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'dimensions_and_pixel_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: dimensions_and_pixel_analysis.png')
print('\n✅ Phase 2 complete!')

---
## 🧹 PHASE 3: DATA PREPROCESSING

Cleaning corrupted/duplicate images, creating data augmentation pipeline.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 3: DATA PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 3: DATA PREPROCESSING')
print('█'*70)

# 3a. Clean dataset - remove corrupted and duplicate files
corrupted_paths = set(c[0] for c in corrupted)
duplicate_names = set(d[0] for d in duplicates)

clean_paths = {}
for cls in class_names:
    clean_paths[cls] = [
        f for f in file_paths[cls]
        if str(f) not in corrupted_paths and f.name not in duplicate_names
    ]

total_clean = sum(len(v) for v in clean_paths.values())
print(f'  Original: {total_images:,} images')
print(f'  Removed: {total_images - total_clean} problematic images')
print(f'  Clean: {total_clean:,} images')

# 3b. Data Augmentation Pipeline
# Why each technique improves model performance:
# - Rotation (±30°): Leaves appear at any angle in nature
# - Shift (±20%): Diseases at different positions on leaf
# - Shear (±15%): Simulates leaf curvature/perspective
# - Zoom (±20%): Varying camera distances
# - Flip (horizontal): Leaf symmetry along vertical axis
# - Brightness (0.8-1.2): Varying lighting conditions (shade, sun, overcast)

data_augmentation = tf.keras.Sequential([
    layers.Resizing(IMG_HEIGHT + 32, IMG_WIDTH + 32),
    layers.RandomRotation(30/360),
    layers.RandomTranslation(0.2, 0.2),
    layers.RandomZoom(0.2),
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(0.2),
    layers.CenterCrop(IMG_HEIGHT, IMG_WIDTH),
    layers.Rescaling(1./255),
])

# Test augmentation on a sample image
sample_img_path = clean_paths[class_names[0]][0]
sample_img = Image.open(sample_img_path)
sample_array = np.array(sample_img, dtype=np.float32)
sample_batch = np.expand_dims(sample_array, 0)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    aug_img = data_augmentation(sample_batch)
    ax.imshow(np.clip(aug_img[0].numpy(), 0, 1))
    ax.axis('off')
    ax.set_title(f'Augmented {i+1}', fontsize=10)

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'augmentation_examples.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Phase 3 complete!')

---
## 📐 PHASE 4: TRAIN / VALIDATION / TEST SPLIT

Stratified 70/15/15 split to maintain class balance across all partitions.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 4: TRAIN/VAL/TEST SPLIT
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 4: DATASET SPLITS')
print('█'*70)

# Create label mapping
label_map = {name: idx for idx, name in enumerate(class_names)}

# Flatten paths and labels
all_paths, all_labels = [], []
for cls in class_names:
    for fpath in clean_paths[cls]:
        all_paths.append(str(fpath))
        all_labels.append(label_map[cls])

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

# First split: training (70%) vs temp (30%)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=all_labels,
)

# Second split: validation (50% of temp) vs test (50% of temp)
# This yields 15% val, 15% test since temp is 30%
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_labels,
)

# Summary
print(f'\n  Training Set   : {len(train_paths):,} images ({len(train_paths)/len(all_paths)*100:.1f}%)')
print(f'  Validation Set : {len(val_paths):,} images ({len(val_paths)/len(all_paths)*100:.1f}%)')
print(f'  Test Set       : {len(test_paths):,} images ({len(test_paths)/len(all_paths)*100:.1f}%)')
print(f'  Total          : {len(all_paths):,} images')

# Show per-class split distribution
split_data = []
for cls_idx, cls_name in enumerate(class_names):
    split_data.append({
        'Class': cls_name[:30],
        'Train': int(np.sum(train_labels == cls_idx)),
        'Val': int(np.sum(val_labels == cls_idx)),
        'Test': int(np.sum(test_labels == cls_idx)),
    })
split_df = pd.DataFrame(split_data)
print('\nPer-class split distribution:')
print(split_df.to_string(index=False))
print('\n✅ Phase 4 complete!')

# Create tf.data.Dataset pipelines
AUTOTUNE = tf.data.AUTOTUNE

def load_and_augment(image_path, label, training=True):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    if training:
        image = data_augmentation(tf.expand_dims(image, 0))
        image = tf.squeeze(image, 0)
    else:
        image = tf.image.resize(image, IMG_SIZE)
        image = image / 255.0
    return image, label

def create_dataset(paths, labels, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: load_and_augment(p, l, training),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels, training=True)
val_ds = create_dataset(val_paths, val_labels, training=False)
test_ds = create_dataset(test_paths, test_labels, training=False)

print('✅ Data pipelines created!')

---
## 🧠 PHASE 5 & 6: MODEL DEVELOPMENT

Building 4 architectures: Custom CNN, MobileNetV2, EfficientNetB0, ResNet50

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 5 & 6: MODEL ARCHITECTURES
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 5 & 6: MODEL ARCHITECTURES')
print('█'*70)

# ─── 5a. Custom CNN ──────────────────────────────────────────────────
def build_custom_cnn(input_shape=INPUT_SHAPE, num_classes=num_classes):
    """
    Custom CNN Architecture:
    ┌──────────────────────────────────────────────┐
    │ Input (224×224×3)                             │
    ├──────────────────────────────────────────────┤
    │ Conv2D(32)+BN+ReLU → MaxPool(2×2) → Drop(0.25)│ → 112×112×32
    │ Conv2D(64)+BN+ReLU → MaxPool(2×2) → Drop(0.25)│ → 56×56×64
    │ Conv2D(128)+BN+ReLU → MaxPool(2×2) → Drop(0.25)│ → 28×28×128
    │ Conv2D(256)+BN+ReLU → MaxPool(2×2) → Drop(0.25)│ → 14×14×256
    ├──────────────────────────────────────────────┤
    │ GlobalAveragePooling2D → Dense(512)+Drop(0.5)  │
    │ Dense(256)+Drop(0.3) → Dense(N)+Softmax        │
    └──────────────────────────────────────────────┘
    ~3.9M parameters
    """
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same',
                          kernel_initializer='he_normal',
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, kernel_initializer='he_normal',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, kernel_initializer='he_normal',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='CustomCNN')
    return model

# ─── 6a. Transfer Learning Helper ─────────────────────────────────────
def add_classification_head(base_model, num_classes):
    """Add GAP + Dense(256) + Dropout(0.5) + Softmax head."""
    base_model.trainable = False
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=base_model.input, outputs=outputs,
                        name=f'{base_model.name}_transfer')

# ─── Build all models ────────────────────────────────────────────────
MODELS = {}

print('\n🔧 Building CustomCNN...')
MODELS['CustomCNN'] = build_custom_cnn()
MODELS['CustomCNN'].summary()

print('\n🔧 Building MobileNetV2...')
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
MODELS['MobileNetV2'] = add_classification_head(base, num_classes)
MODELS['MobileNetV2'].summary()

print('\n🔧 Building EfficientNetB0...')
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
MODELS['EfficientNetB0'] = add_classification_head(base, num_classes)
MODELS['EfficientNetB0'].summary()

print('\n🔧 Building ResNet50...')
base = ResNet50(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
MODELS['ResNet50'] = add_classification_head(base, num_classes)
MODELS['ResNet50'].summary()

print('✅ All models built successfully!')

---
## 🏋️ PHASE 7: MODEL TRAINING

Training all models with Early Stopping, ReduceLROnPlateau, and ModelCheckpoint.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 7: MODEL TRAINING
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 7: MODEL TRAINING')
print('█'*70)

all_training_history = {}
all_models = {}  # best model instance per architecture

# Select which models to train (1 or 2 for demo; all 4 for full run)
models_to_train = ['CustomCNN', 'MobileNetV2', 'EfficientNetB0', 'ResNet50']
# models_to_train = ['CustomCNN', 'MobileNetV2']  # Faster demo option

for model_name in models_to_train:
    print(f'\n{"█"*70}')
    print(f'█  TRAINING: {model_name}')
    print(f'{"█"*70}')
    
    # Get model
    model = MODELS[model_name]
    
    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    
    # Callbacks
    checkpoint_path = str(MODELS_DIR / f'{model_name}_best.weights.h5')
    csv_log_path = str(LOGS_DIR / f'{model_name}_log.csv')
    
    callbacks_list = [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, verbose=1, mode='min'
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=checkpoint_path, monitor='val_accuracy',
            save_best_only=True, save_weights_only=True, mode='max'
        ),
        callbacks.CSVLogger(filename=csv_log_path, separator=',', append=False),
    ]
    
    # Train
    start_time = time.time()
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=callbacks_list, verbose=1
    )
    training_time = time.time() - start_time
    
    all_training_history[model_name] = history.history
    all_models[model_name] = model
    
    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history.history['accuracy']) + 1)
    
    axes[0].plot(epochs, history.history['accuracy'], 'b-', label='Train Acc', linewidth=2)
    axes[0].plot(epochs, history.history['val_accuracy'], 'r-', label='Val Acc', linewidth=2)
    axes[0].set_title(f'{model_name} - Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epochs'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, history.history['loss'], 'b-', label='Train Loss', linewidth=2)
    axes[1].plot(epochs, history.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[1].set_title(f'{model_name} - Loss', fontweight='bold')
    axes[1].set_xlabel('Epochs'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / f'{model_name}_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'  ✅ {model_name} trained! Time: {training_time:.1f}s')
    print(f'     Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')

print('\n✅ Phase 7 complete!')

---
## 📈 PHASE 8: MODEL EVALUATION

Comprehensive evaluation with accuracy, precision, recall, F1, confusion matrices, ROC curves.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 8: MODEL EVALUATION
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 8: MODEL EVALUATION')
print('█'*70)

all_results = {}

for model_name in models_to_train:
    if model_name not in all_models:
        continue
    model = all_models[model_name]
    
    print(f'\n{"="*70}')
    print(f'📊 Evaluating {model_name}')
    print(f'{"="*70}')
    
    # Get predictions
    y_prob = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    
    # Metrics
    acc = accuracy_score(test_labels, y_pred)
    prec = precision_score(test_labels, y_pred, average='macro', zero_division=0)
    rec = recall_score(test_labels, y_pred, average='macro', zero_division=0)
    f1 = f1_score(test_labels, y_pred, average='macro', zero_division=0)
    
    all_results[model_name] = {
        'accuracy': float(acc),
        'precision_macro': float(prec),
        'recall_macro': float(rec),
        'f1_macro': float(f1),
    }
    
    print(f'\n  Accuracy          : {acc:.4f}')
    print(f'  Precision (macro) : {prec:.4f}')
    print(f'  Recall (macro)    : {rec:.4f}')
    print(f'  F1-Score (macro)  : {f1:.4f}')
    
    # Per-class metrics
    print('\n  Per-class performance:')
    print(f'  {"Class":<40} {"Prec":<8} {"Rec":<8} {"F1":<8}')
    print(f'  {"-"*64}')
    
    report = classification_report(test_labels, y_pred,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
    for cls in class_names:
        if cls in report:
            print(f'  {cls:<40} {report[cls]["precision"]:.3f}  '
                  f'{report[cls]["recall"]:.3f}  {report[cls]["f1-score"]:.3f}')
    
    # Confusion Matrix
    cm = confusion_matrix(test_labels, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
    
    short_names = [c.replace('___', '\n').replace('_', ' ') for c in class_names]
    
    fig, ax = plt.subplots(figsize=(16, 14))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names,
                ax=ax, vmin=0, vmax=1, linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': 'Proportion'})
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('True', fontsize=14)
    ax.set_title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / f'{model_name}_confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # ROC Curves (one-vs-rest)
    n_classes = num_classes
    y_true_bin = np.zeros((len(test_labels), n_classes))
    for i in range(n_classes):
        y_true_bin[:, i] = (test_labels == i).astype(int)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    colors = plt.cm.viridis(np.linspace(0, 1, n_classes))
    
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f'{class_names[i][:25]} (AUC={roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC=0.500)')
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title(f'{model_name} - ROC Curves', fontsize=15, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / f'{model_name}_roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('\n✅ Phase 8 complete!')

---
## 🔍 PHASE 9: ERROR ANALYSIS

Investigating misclassified images to understand model limitations.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 9: ERROR ANALYSIS
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 9: ERROR ANALYSIS')
print('█'*70)

for model_name in models_to_train:
    if model_name not in all_models:
        continue
    model = all_models[model_name]
    
    print(f'\n{"="*70}')
    print(f'🔍 Error Analysis: {model_name}')
    print(f'{"="*70}')
    
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    
    misclassified = np.where(test_labels != y_pred)[0]
    print(f'  Total misclassified: {len(misclassified)} / {len(test_labels)} '
          f'({len(misclassified)/len(test_labels)*100:.2f}%)')
    
    # Per-class error rates
    print(f'\n  Per-class Error Rates:')
    error_data = []
    for i, cls in enumerate(class_names):
        class_idx = np.where(test_labels == i)[0]
        if len(class_idx) > 0:
            errors = np.sum(y_pred[class_idx] != test_labels[class_idx])
            error_rate = errors / len(class_idx)
            error_data.append({'class': cls, 'errors': errors, 'rate': error_rate})
    
    error_df = pd.DataFrame(error_data).sort_values('rate', ascending=False)
    print(error_df.to_string(index=False))
    
    # Top confusion pairs
    print(f'\n  Top Confusion Pairs (True → Predicted):')
    confusion_pairs = {}
    for idx in misclassified:
        pair = (class_names[test_labels[idx]], class_names[y_pred[idx]])
        confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1
    
    sorted_pairs = sorted(confusion_pairs.items(), key=lambda x: -x[1])[:10]
    for (true_cls, pred_cls), count in sorted_pairs:
        print(f'    {true_cls[:35]:<35} → {pred_cls[:35]:<35} [{count}x]')
    
    # Show misclassified examples
    if len(misclassified) > 0:
        n_examples = min(12, len(misclassified))
        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        axes = axes.flatten()
        
        for i in range(n_examples):
            idx = misclassified[i]
            try:
                img = Image.open(test_paths[idx])
                axes[i].imshow(img)
                true_name = class_names[test_labels[idx]].replace('_', ' ')
                pred_name = class_names[y_pred[idx]].replace('_', ' ')
                axes[i].set_title(f'True: {true_name[:18]}\nPred: {pred_name[:18]}',
                                 fontsize=8, color='red')
            except:
                axes[i].text(0.5, 0.5, 'Error', ha='center', va='center')
            axes[i].axis('off')
        
        for i in range(n_examples, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle(f'{model_name} - Misclassified Examples', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(str(PLOTS_DIR / f'{model_name}_error_analysis.png'), dpi=150)
        plt.show()

print('\n✅ Phase 9 complete!')

---
## 🏆 MODEL COMPARISON

Comparing all trained models side by side.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MODEL COMPARISON
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  MODEL COMPARISON')
print('█'*70)

# Display comparison table
comparison_df = pd.DataFrame(all_results).T
print('\n📊 Model Performance Comparison:')
print(comparison_df.round(4).to_string())

# Bar chart comparison
model_names = list(all_results.keys())
metrics_to_plot = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(model_names))
width = 0.2
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

for i, metric in enumerate(metrics_to_plot):
    values = [all_results[m].get(metric, 0) for m in model_names]
    bars = ax.bar(x + i * width, values, width,
                  label=metric.replace('_', ' ').title(),
                  color=colors[i], edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# Identify best model
best_model = max(all_results, key=lambda m: all_results[m]['f1_macro'])
print(f'\n🏆 Best overall model: {best_model} (F1-Score: {all_results[best_model]["f1_macro"]:.4f})')

# Save results
with open(RESULTS_DIR / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('✅ Results saved to results/all_results.json')

---
## 🚀 PHASE 10: INFERENCE SYSTEM

Predict disease from new leaf images using the best trained model.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PHASE 10: INFERENCE SYSTEM
# ═══════════════════════════════════════════════════════════════════════

print('█'*70)
print('█  PHASE 10: INFERENCE SYSTEM')
print('█'*70)

def preprocess_for_inference(image_path):
    """Load and preprocess a single image for model prediction."""
    img = Image.open(image_path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.Resampling.LANCZOS)
    img_array = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(img_array, axis=0)

def predict_disease(image_path, model, model_name='Best Model'):
    """
    Predict disease from a leaf image.
    
    Returns:
        Dictionary with prediction class, confidence, and top-3 predictions
    """
    img_array = preprocess_for_inference(image_path)
    predictions = model.predict(img_array, verbose=0)[0]
    
    predicted_idx = np.argmax(predictions)
    confidence = float(predictions[predicted_idx])
    
    top_3_idx = np.argsort(predictions)[::-1][:3]
    top_3 = [
        {'class': class_names[i], 'confidence': float(predictions[i])}
        for i in top_3_idx
    ]
    
    return {
        'predicted_class': class_names[predicted_idx],
        'confidence': confidence,
        'top_3': top_3,
    }

# Demo: Test on a few test set images
print(f'\n🔬 Testing inference on test set samples using "{best_model}"...')
best_model_instance = all_models[best_model]

# Get test image paths
test_image_paths = test_paths[:12]  # First 12 test images
test_image_labels = test_labels[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, (img_path, true_label) in enumerate(zip(test_image_paths, test_image_labels)):
    result = predict_disease(img_path, best_model_instance, best_model)
    
    img = Image.open(img_path)
    axes[i].imshow(img)
    
    true_name = class_names[true_label].replace('_', ' ')
    pred_name = result['predicted_class'].replace('_', ' ')
    
    is_correct = true_label == class_names.index(result['predicted_class'])
    color = 'green' if is_correct else 'red'
    
    axes[i].set_title(f'True: {true_name[:20]}\nPred: {pred_name[:20]}\nConf: {result["confidence"]:.2%}',
                     fontsize=9, color=color, fontweight='bold')
    axes[i].axis('off')

plt.suptitle(f'{best_model} - Inference Demo on Test Samples',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'inference_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save the best model for production use
model_save_path = str(MODELS_DIR / f'{best_model}_final.h5')
best_model_instance.save(model_save_path)
print(f'✅ Best model saved to: {model_save_path}')
print('\n✅ Phase 10 complete!')

---
## 📋 PREDICT FROM YOUR OWN IMAGE

Run this cell to predict disease from any leaf image by uploading or providing a file path.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📸 CUSTOM PREDICTION CELL
# ═══════════════════════════════════════════════════════════════════════
# To use: set IMAGE_PATH below to your image file

IMAGE_PATH = None  # ← Set this to your image path, e.g. "path/to/leaf.jpg"

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    result = predict_disease(IMAGE_PATH, best_model_instance, best_model)
    
    print('='*60)
    print('🌿 LEAF DISEASE PREDICTION')
    print('='*60)
    print(f'  Predicted Disease: {result["predicted_class"]}')
    print(f'  Confidence: {result["confidence"]:.2%}')
    print('-'*60)
    print('  Top-3 Predictions:')
    for i, pred in enumerate(result['top_3'], 1):
        print(f'    {i}. {pred["class"]:40s} {pred["confidence"]:.2%}')
    print('='*60)
    
    # Display image with prediction
    img = Image.open(IMAGE_PATH)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Prediction: {result["predicted_class"]}\nConfidence: {result["confidence"]:.2%}',
              fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()
else:
    print('📝 Set IMAGE_PATH to your leaf image to get a prediction.')
    print('Example:')
    print('  IMAGE_PATH = "path/to/your/leaf_photo.jpg"')

---
## 📊 RESULTS SUMMARY

| Model | Accuracy | Precision | Recall | F1-Score |
|-------|----------|-----------|--------|----------|
| CustomCNN | - | - | - | - |
| MobileNetV2 | - | - | - | - |
| EfficientNetB0 | - | - | - | - |
| ResNet50 | - | - | - | - |

*Results will populate after running the evaluation cells above.*

## 🗂️ Output Files

All generated files are saved to:
- **Plots**: `results/plots/`
- **Models**: `results/models/`
- **Logs**: `results/logs/`
- **Results JSON**: `results/all_results.json`

---

*End of Complete Pipeline*